In [1]:
from pathlib import Path
import os
DATA_PATH = Path(os.getcwd()).parent / "data"
RAW_PATH = DATA_PATH / "raw"

In [ ]:
bag_files = list(RAW_PATH.glob("*.bag"))
bag_files

[PosixPath('/mnt/pool-1/projects/ubuntu/AIT-brainlab/sign-language/data/raw/2-m.bag'),
 PosixPath('/mnt/pool-1/projects/ubuntu/AIT-brainlab/sign-language/data/raw/1-m.bag'),
 PosixPath('/mnt/pool-1/projects/ubuntu/AIT-brainlab/sign-language/data/raw/1-r.bag'),
 PosixPath('/mnt/pool-1/projects/ubuntu/AIT-brainlab/sign-language/data/raw/2-r.bag')]

In [75]:
bag_file = bag_files[0]
bag_file

PosixPath('/mnt/pool-1/projects/ubuntu/AIT-brainlab/sign-language/data/raw/2-m.bag')

In [82]:
from datetime import datetime
def process(pipeline, history, log_file_name):
    try:
        log_file = open(log_file_name, "w")
        fps = pipeline.get_active_profile().get_stream(rs.stream.color).as_video_stream_profile().fps()
        frame_in_second = 0
        current_second = 0
        while True:
            # Wait for the next set of frames
            frames = pipeline.wait_for_frames()

            # Retrieve specific streams
            depth_frame = frames.get_depth_frame()
            color_frame = frames.get_color_frame()
            
            # Retrieve IMU frames (if they were recorded)
            accel_frame = frames.first_or_default(rs.stream.accel)
            gyro_frame = frames.first_or_default(rs.stream.gyro)

            frame_tz = frames.get_timestamp()
            frame_time = datetime.fromtimestamp(frame_tz / 1000.0)
            frame_number = frames.get_frame_number()
            second = frame_time.second
            if second != current_second:
                history.append(frame_in_second)
                current_second = second
                frame_in_second = 0
            else:
                frame_in_second += 1
            log_file.write(f"FPS: {fps}, Frame Rate: {frame_in_second:03d}, Frame Number: {frame_number}, Timestamp: {frame_tz}, Time: {frame_time}\n")
            # Process Color Frame
            if color_frame:
                color_image = np.asanyarray(color_frame.get_data())
                color_image_bgr = cv2.cvtColor(color_image, cv2.COLOR_RGB2BGR)
                cv2.imshow("Color Stream", color_image_bgr)

            # Process Depth Frame
            if depth_frame:
                # print(frames.get_frame_metadata(depth_frame))
                depth_image = np.asanyarray(depth_frame.get_data())
                # Normalize depth to 8-bit for visualization
                depth_image_8bit = cv2.convertScaleAbs(depth_image, alpha=0.03)
                depth_colormap = cv2.applyColorMap(depth_image_8bit, cv2.COLORMAP_JET)
                cv2.imshow("Depth Stream", depth_colormap)

            # Process IMU Data
            if accel_frame:
                accel_data = accel_frame.as_motion_frame().get_motion_data()
                print(f"Accel: x={accel_data.x:.2f}, y={accel_data.y:.2f}, z={accel_data.z:.2f}")

            if gyro_frame:
                gyro_data = gyro_frame.as_motion_frame().get_motion_data()
                print(f"Gyro: x={gyro_data.x:.2f}, y={gyro_data.y:.2f}, z={gyro_data.z:.2f}")

            # Exit on pressing 'q'
            key = cv2.waitKey(1) & 0xFF
            if key == ord("q"):
                break

    finally:
        pipeline.stop()
        cv2.destroyAllWindows()
        log_file.close()

In [89]:
import pyrealsense2 as rs
import numpy as np
import cv2
bag_file = bag_files[3]
# Configure playback from bag file
pipeline = rs.pipeline()
config = rs.config()
config.enable_device_from_file(str(bag_file), repeat_playback=False)

profile = pipeline.start(config)

# Set the real-time flag to false for synchronous playback and frame grabbing
device = profile.get_device()
playback = device.as_playback()
# Real-Time Playback: The playback.set_real_time(False) line is critical. 
# It forces the pipeline to step through the frames as fast as your code allows, 
# rather than matching the real-world timestamp rate of the hardware recording.
playback.set_real_time(False) 
history = []
process(pipeline, history, bag_file.stem + "_frame_log.txt")
pipeline.stop()


RuntimeError: Frame didn't arrive within 5000

In [90]:
with(open(bag_file.stem + "_frame_log.txt", "a") as log_file):
    log_file.write("\nFrame Rate History (Frames per Second):\n")
    log_file.write(f"{history}")